<a href="https://colab.research.google.com/github/hemachandranrcs24-spec/BIS_LAB/blob/main/week41bm24cs115_hemachandranr_parallelcellalgorithim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import math
from concurrent.futures import ThreadPoolExecutor

# ==========================================
# 1. SETUP: RIDERS, ORDERS & CELL STRUCTURE
# ==========================================
class Cell:
    def __init__(self, row, col, assignment):
        self.row = row
        self.col = col
        # assignment is a list where index = order_id, value = rider_id
        self.assignment = assignment
        self.fitness = float('inf')

# Generate sample 2D coordinates (x, y) for riders and orders
NUM_RIDERS = 5
NUM_ORDERS = 8
GRID_ROWS, GRID_COLS = 4, 4  # 16 parallel cells

random.seed(42)
rider_locations = {r: (random.uniform(0, 20), random.uniform(0, 20)) for r in range(NUM_RIDERS)}
rider_availability = {r: random.uniform(0.5, 1.0) for r in range(NUM_RIDERS)} # 1.0 = fully free
order_locations = {o: (random.uniform(0, 20), random.uniform(0, 20)) for o in range(NUM_ORDERS)}

def calculate_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

# ==========================================
# 2. OBJECTIVE 3: EVALUATE FITNESS
# ==========================================
def evaluate_fitness(cell):
    """
    Calculates fitness based on total delivery distance and rider workload/availability.
    Lower fitness score = better allocation.
    """
    total_distance = 0.0
    rider_order_count = {r: 0 for r in range(NUM_RIDERS)}

    for order_id, rider_id in enumerate(cell.assignment):
        dist = calculate_distance(rider_locations[rider_id], order_locations[order_id])
        # Factor in rider availability (less available riders incur a time penalty)
        effective_time = dist / rider_availability[rider_id]
        total_distance += effective_time
        rider_order_count[rider_id] += 1

    # Add workload penalty if a single rider gets overloaded with too many orders
    workload_penalty = sum((count ** 2) * 2.5 for count in rider_order_count.values())

    cell.fitness = total_distance + workload_penalty
    return cell.fitness

# ==========================================
# 3. OBJECTIVE 4: NEIGHBORHOOD & UPDATE RULE
# ==========================================
def get_moore_neighbors(grid, r, c):
    """Finds 8 surrounding neighbors in the cellular grid."""
    neighbors = []
    for dr in [-1, 0, 1]:
        for dc in [-1, 0, 1]:
            if dr == 0 and dc == 0:
                continue
            nr, nc = (r + dr) % GRID_ROWS, (c + dc) % GRID_COLS  # Wrap-around grid
            neighbors.append(grid[nr][nc])
    return neighbors

def update_cell_state(grid, r, c, alpha=0.5):
    """Updates a cell's rider-order assignment using the best neighboring cell."""
    current_cell = grid[r][c]
    neighbors = get_moore_neighbors(grid, r, c)

    # Find neighbor with the lowest fitness (minimum delivery time)
    best_neighbor = min(neighbors, key=lambda cell: cell.fitness)

    new_assignment = list(current_cell.assignment)
    if best_neighbor.fitness < current_cell.fitness:
        # Adopt rider assignments from the better neighbor based on rate alpha
        for i in range(NUM_ORDERS):
            if random.random() < alpha:
                new_assignment[i] = best_neighbor.assignment[i]

    # Small random mutation (10% chance) to explore new rider allocations
    if random.random() < 0.1:
        rand_order = random.randint(0, NUM_ORDERS - 1)
        new_assignment[rand_order] = random.randint(0, NUM_RIDERS - 1)

    return new_assignment

# ==========================================
# 4. MAIN PARALLEL CELLULAR ALGORITHM
# ==========================================
def run_pca_delivery_allocation(max_iterations=50):
    # Objective 1 & 2: Initialize grid of cells with random rider-order assignments
    grid = []
    for r in range(GRID_ROWS):
        row_cells = []
        for c in range(GRID_COLS):
            random_assignment = [random.randint(0, NUM_RIDERS - 1) for _ in range(NUM_ORDERS)]
            row_cells.append(Cell(r, c, random_assignment))
        grid.append(row_cells)

    best_overall_cell = None

    # Use ThreadPoolExecutor to simulate parallel cell execution
    with ThreadPoolExecutor() as executor:
        for iteration in range(max_iterations):
            all_cells = [cell for row in grid for cell in row]

            # Step A: Evaluate fitness of all cells in parallel
            list(executor.map(evaluate_fitness, all_cells))

            # Track the global best allocation
            current_best = min(all_cells, key=lambda cell: cell.fitness)
            if best_overall_cell is None or current_best.fitness < best_overall_cell.fitness:
                best_overall_cell = Cell(current_best.row, current_best.col, list(current_best.assignment))
                best_overall_cell.fitness = current_best.fitness

            # Step B: Update all cell states simultaneously using neighbor data
            futures = {}
            for r in range(GRID_ROWS):
                for c in range(GRID_COLS):
                    futures[(r, c)] = executor.submit(update_cell_state, grid, r, c)

            # Apply updated states synchronously for the next iteration (t -> t+1)
            for (r, c), future in futures.items():
                grid[r][c].assignment = future.result()

            if iteration % 10 == 0 or iteration == max_iterations - 1:
                print(f"Iteration {iteration:02d} | Best Fitness (Delivery Cost): {best_overall_cell.fitness:.2f}")

    return best_overall_cell

# ==========================================
# 5. OBJECTIVE 5: OUTPUT OPTIMAL ALLOCATION
# ==========================================
if __name__ == "__main__":
    print("Starting Parallel Cellular Algorithm for Food Delivery Allocation...\n")
    optimal_solution = run_pca_delivery_allocation(max_iterations=50)

    print("\n--- OPTIMAL RIDER-ORDER ALLOCATION ---")
    print(f"Minimum Delivery Cost Score: {optimal_solution.fitness:.2f}\n")
    for order_id, rider_id in enumerate(optimal_solution.assignment):
        dist = calculate_distance(rider_locations[rider_id], order_locations[order_id])
        print(f"Order #{order_id} -> Assigned to Rider #{rider_id} (Distance: {dist:.2f} km, Availability: {rider_availability[rider_id]:.2f})")


Starting Parallel Cellular Algorithm for Food Delivery Allocation...

Iteration 00 | Best Fitness (Delivery Cost): 149.18
Iteration 10 | Best Fitness (Delivery Cost): 149.18
Iteration 20 | Best Fitness (Delivery Cost): 149.18
Iteration 30 | Best Fitness (Delivery Cost): 149.18
Iteration 40 | Best Fitness (Delivery Cost): 140.05
Iteration 49 | Best Fitness (Delivery Cost): 138.70

--- OPTIMAL RIDER-ORDER ALLOCATION ---
Minimum Delivery Cost Score: 138.70

Order #0 -> Assigned to Rider #0 (Distance: 4.34 km, Availability: 0.61)
Order #1 -> Assigned to Rider #2 (Distance: 3.96 km, Availability: 0.51)
Order #2 -> Assigned to Rider #1 (Distance: 12.83 km, Availability: 0.75)
Order #3 -> Assigned to Rider #4 (Distance: 8.31 km, Availability: 0.82)
Order #4 -> Assigned to Rider #4 (Distance: 19.30 km, Availability: 0.82)
Order #5 -> Assigned to Rider #1 (Distance: 2.89 km, Availability: 0.75)
Order #6 -> Assigned to Rider #1 (Distance: 12.98 km, Availability: 0.75)
Order #7 -> Assigned to Rid